# BantAI — Fine-tune XLM-RoBERTa

**Sprint 2 · Track B (AI/ML) · WBS 2.3.4**

Trains the smishing classifier on the labeled dataset (Ham / Spam / Scam)
and produces `models/xlm-roberta-smishing/`, which the FastAPI inference
service loads automatically.

### Before you start
1. **Runtime → Change runtime type → T4 GPU** (free tier is enough).
2. Have `bantai_colab_package.zip` ready to upload (step 3).

Expected total runtime on a T4: **~20-30 minutes.**

> **Re-running after a previous attempt?** Just run the cells again — step 3
> now deletes any package left over from the earlier run before uploading,
> and step 5 refuses to train if the split still contains leakage.


## 1. Confirm a GPU is attached


In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU attached. Go to Runtime > Change runtime type > T4 GPU, '
        'then re-run this cell. (Training on CPU would take many hours.)'
    )

print('GPU :', torch.cuda.get_device_name(0))
print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))


## 2. Install dependencies

Colab already ships torch / pandas / numpy / scikit-learn, so only the
HuggingFace stack is installed here. `transformers>=4.46` is required — that
is the release that renamed `Trainer(tokenizer=...)` to `processing_class`.


In [ ]:
%pip install -q 'transformers>=4.46' 'datasets>=2.19' accelerate sentencepiece

import transformers, datasets
print('transformers', transformers.__version__)
print('datasets    ', datasets.__version__)


## 3. Upload the package

Run the cell, then pick **`bantai_colab_package.zip`** from your computer
(it lives in `ai/colab/` in the repo).

Colab does **not** overwrite an existing file — a second upload of the same
name arrives as `bantai_colab_package (1).zip`. So any earlier copy is
deleted first, and the filename actually returned by the picker is what gets
unpacked below. Otherwise a re-run silently trains on the stale package.


In [ ]:
import glob
import os

from google.colab import files

for stale in glob.glob('/content/bantai_colab_package*.zip'):
    os.remove(stale)
    print('removed stale upload:', stale)

uploaded = files.upload()
PACKAGE = '/content/' + list(uploaded)[0]
print('\nwill unpack:', PACKAGE)


## 4. Unpack and verify the split


In [ ]:
import shutil
import zipfile

shutil.rmtree('/content/bantai_ai', ignore_errors=True)
with zipfile.ZipFile(PACKAGE) as z:
    z.extractall('/content/bantai_ai')
%cd /content/bantai_ai

import sys

sys.path.insert(0, '.')

from training.config import ID2LABEL, TrainingConfig
from training.dataset import load_split

train_texts, val_texts, train_labels, val_labels = load_split(TrainingConfig())
print('train rows:', len(train_texts))
print('val rows  :', len(val_texts))

# Messages are de-duplicated on their *masked* form, because that is what the
# model sees: '...libre 1q2w3e7.ca' and '...libre 1q2w3e8.ca' both become
# '...libre <URL>'. Without this the same string lands on both sides of the
# split and the model is scored on text it memorised. Fail fast rather than
# spend 25 minutes producing an inflated number.
overlap = set(val_texts) & set(train_texts)
if overlap:
    raise SystemExit(
        f'LEAKAGE: {len(overlap)} validation messages also appear in training. '
        'You uploaded an outdated package -- re-run step 3 with the current '
        'bantai_colab_package.zip from ai/colab/.'
    )
print('\nleakage check: PASS (no shared messages)')
print('expected roughly 10,867 train / 2,717 val')

from collections import Counter

print('val class balance:',
      {ID2LABEL[k]: v for k, v in sorted(Counter(val_labels).items())})


## 5. Fine-tune

4 epochs over a stratified 80/20 split, AdamW at lr 2e-5, batch size 16,
max 128 tokens, fp16 on GPU.

Accuracy, precision, recall and macro-F1 are reported after every epoch, and
the **best epoch by macro-F1** is what gets saved — not simply the last one.

> Watch macro-F1, not accuracy. Ham is ~62% of the data, so a model that
> guessed "Ham" every single time would still score ~62% accuracy while being
> completely useless. Macro-F1 weights all three classes equally.


In [ ]:
!python -m training.train


## 6. Per-class results + confusion matrix

Re-creates the exact same validation split (fixed seed 42) and scores the
saved model on it. These numbers go straight into the thesis — they also
cover the false-positive / false-negative matrix in WBS 6.4.6.

**Sanity check:** the `support` column must sum to the val row count printed
in step 4. If it says 2986, an outdated package was used — go back to step 3.


In [ ]:
import torch
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer

cfg = TrainingConfig()
tok = AutoTokenizer.from_pretrained(cfg.output_dir)
model = AutoModelForSequenceClassification.from_pretrained(cfg.output_dir)
model = model.cuda().eval()

preds = []
with torch.no_grad():
    for i in range(0, len(val_texts), 64):
        batch = tok(
            val_texts[i:i + 64],
            truncation=True,
            max_length=cfg.max_length,
            padding=True,
            return_tensors='pt',
        ).to('cuda')
        preds.extend(model(**batch).logits.argmax(-1).cpu().tolist())

names = [ID2LABEL[i] for i in range(cfg.num_labels)]
print(classification_report(val_labels, preds, target_names=names, digits=4))
print('Confusion matrix (rows = actual, cols = predicted)')
print('        ' + ''.join('%8s' % n for n in names))
for name, row in zip(names, confusion_matrix(val_labels, preds)):
    print('%-8s' % name + ''.join('%8d' % v for v in row))


## 7. Save the trained model

The model is ~1.1 GB. Saving to Google Drive is the reliable route; the
direct browser download in the next cell often stalls at this size.

**Colab deletes everything when the session ends — do not skip this step.**


In [ ]:
!zip -qr /content/bantai_model.zip models/xlm-roberta-smishing
!ls -lh /content/bantai_model.zip

from google.colab import drive

drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/bantai'
!cp /content/bantai_model.zip '/content/drive/MyDrive/bantai/'
print('Saved to Google Drive: MyDrive/bantai/bantai_model.zip')


Optional direct download instead of / as well as Drive:


In [ ]:
from google.colab import files

files.download('/content/bantai_model.zip')


## 8. Back on your laptop

Unzip the downloaded file into the repo so the inference service picks it up:

```
ai/models/xlm-roberta-smishing/
```

Then start the service and check that it reports the model as ready:

```bash
cd ai
.venv/Scripts/uvicorn service.main:app --port 8001
# GET /health  ->  {"status": "ok", "model_ready": true}
```

`ai/models/` is git-ignored, so the weights stay out of the repo.
